# DEPARTMENT OF COMPUTER SCIENCE
## UNIVERSITY OF ENGINEERING & TECHNOLOGY, PESHAWAR
### Data Science Lab # 10 & 11: Singular Value Decomposition (SVD) and Principal Component Analysis (PCA)

### Software Requirements & Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD

# Plotting style configuration
plt.style.use('ggplot')

## Part A: Data Preprocessing

### Task 1: Load the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('Feature_Engineering_Dataset_100Rows.xlsx - Realistic_Student_Data.csv')

#### Deliverable: Display first 5 rows

In [ ]:
df.head()

#### Deliverable: Display dataset shape

In [ ]:
print(f'Dataset Shape (Rows, Columns): {df.shape}')

#### Deliverable: Display data types

In [ ]:
df.dtypes

### Task 2: Encode Categorical Variables (Label Encoding)

In [ ]:
# Initialize LabelEncoders
le_gender = LabelEncoder()
le_dept = LabelEncoder()
le_result = LabelEncoder()

# Apply encoding to categorical features
df['Gender'] = le_gender.fit_transform(df['Gender'])
df['Department'] = le_dept.fit_transform(df['Department'])
df['Final_Result'] = le_result.fit_transform(df['Final_Result'])

#### Deliverable: Display transformed dataset

In [ ]:
df.head()

### Task 3: Feature Scaling using StandardScaler()

In [ ]:
# Separate features for dimensionality reduction (dropping unique identifier Student_ID)
X = df.drop(columns=['Student_ID'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#### Verification of Scaled Data (Mean = 0, Std = 1)

In [ ]:
print('Feature Means after Standardization:\n', np.round(X_scaled.mean(axis=0), 2))
print('\nFeature Standard Deviations after Standardization:\n', np.round(X_scaled.std(axis=0), 2))

### Questions & Answers
**1. Why is feature scaling required before PCA?**
PCA is highly sensitive to the variances of the original variables. Features with much larger scales/ranges will dominate the component axes, leading to a biased projection that ignores lower-magnitude features regardless of their informational value.

**2. What problems can occur if scaling is skipped?**
If skipped, variables like `Family_Income` would completely overwhelm variables with small ranges like `Previous_GPA`. The principal components will simply align with the features holding the highest absolute variance rather than capturing genuine multidimensional structural relationships.

## Part B: Singular Value Decomposition (SVD)

### Task 4: Apply Singular Value Decomposition (Reduce to Two Components)

In [ ]:
svd = TruncatedSVD(n_components=2, random_state=42)
X_svd = svd.fit_transform(X_scaled)

#### Deliverable: Display transformed data matrix

In [ ]:
svd_df = pd.DataFrame(data=X_svd, columns=['Component_1', 'Component_2'])
svd_df['Final_Result'] = df['Final_Result']
svd_df.head()

### Task 5: Analyze Singular Values

In [ ]:
print('SVD Singular Values:', svd.singular_values_)

### Questions & Answers
**1. What does a singular value represent?**
A singular value represents the square root of the eigenvalues of the data matrix multiplied by its transpose. It signifies the geometric magnitude, spread, or structural strength of the dataset along that particular singular vector component axis.

**2. Which component captures the most information?**
Component 1 captures the most information because components are ordered in descending order of their corresponding singular values. The first component aligns with the path of maximum data variance.

### Task 6: SVD Visualization

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(svd_df['Component_1'], svd_df['Component_2'], 
                      c=svd_df['Final_Result'], cmap='coolwarm', alpha=0.8, edgecolors='w')
plt.title('SVD - 2D Projection of Student Data', fontsize=14)
plt.xlabel('SVD Component 1')
plt.ylabel('SVD Component 2')
plt.legend(handles=scatter.legend_elements()[0], labels=['Fail', 'Pass'], title='Final Result')
plt.show()

#### Brief Interpretation:
The SVD plot maps the 7-dimensional student dataset into 2 dimensions. Along SVD Component 1, we can observe a distinct visual gradient where students who failed cluster predominantly on the left and center, while students who passed spread out toward the right. This indicates that the top singular vector accurately captures the underlying variations tied to academic outcomes.

## Part C: Principal Component Analysis (PCA)

### Task 7: Apply PCA (Reduce to Two Principal Components)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

#### Deliverable: Display transformed dataset

In [ ]:
pca_df = pd.DataFrame(data=X_pca, columns=['PC1', 'PC2'])
pca_df['Final_Result'] = df['Final_Result']
pca_df.head()

### Task 8: Variance Analysis

In [ ]:
explained_variance = pca.explained_variance_ratio_
print('Explained Variance Ratio per Component:', explained_variance)

#### Calculate Total Variance Explained

In [ ]:
total_variance = np.sum(explained_variance) * 100
print(f'Total Variance Retained by PC1 and PC2: {total_variance:.2f}%')

### Questions & Answers
**1. Which principal component explains the most variance?**
**PC1 (Principal Component 1)** explains the most variance, capturing the maximum global spread of the dataset.

**2. How much total variance is retained?**
The total variance retained by PC1 and PC2 combined captures more than half of the entire dataset's informational layout (exact numerical value is printed in the execution cell above).

### Task 9: PCA Visualization

In [ ]:
plt.figure(figsize=(8, 6))
scatter_pca = plt.scatter(pca_df['PC1'], pca_df['PC2'], 
                          c=pca_df['Final_Result'], cmap='bwr', alpha=0.8, edgecolors='k')
plt.title('PCA - 2D Projection of Student Data', fontsize=14)
plt.xlabel('Principal Component 1 (PC1)')
plt.ylabel('Principal Component 2 (PC2)')
plt.legend(handles=scatter_pca.legend_elements()[0], labels=['Fail', 'Pass'], title='Final Result')
plt.show()

#### Interpretation:
The 2D PCA representation successfully segregates the target outcomes. Students who passed are mapped towards the right side of the graph (high positive values of PC1), which means PC1 is strongly correlated with features indicating performance (like Study Hours, Attendance, and Previous GPA). Conversely, students who failed occupy the left zone.

## Part D: Comparison of SVD and PCA

### Task 10: Comparison Summary Table

| Feature | SVD (Truncated SVD) | PCA (Principal Component Analysis) |
| :--- | :--- | :--- |
| **Purpose** | Factors matrices directly; ideal for structural matrices. | Maximizes variance to find orthogonal projection patterns. |
| **Output Components** | Left/Right Singular Vectors & Singular Values. | Principal Components (Eigenvectors) & Eigenvalues. |
| **Variance Information** | Indirectly optimized through singular values. | Explicitly maximized and ordered by explained variance ratio. |
| **Visualization Quality** | Good, but identical to PCA only if data is centered. | Excellent; extracts maximum possible orthogonal scatter spread. |
| **Computational Cost** | Highly efficient on large, sparse datasets. | Slightly more costly due to explicit centering & covariance operations. |

### Final Conclusion:
Because the dataset features were mean-centered and standardized using `StandardScaler`, **PCA and Truncated SVD deliver mathematically equivalent projection configurations**. This lab confirms how 7-dimensional profiles can be successfully summarized down to 2 axes without omitting predictive behavior signals.